# Jev-CeNN | V3.1-style preservation-first: FAST top-layer attention-only training

**What changed:** Train only the selected top encoder layer's attention + CeNN pathway. Preserve all 28 encoder slots, pretrained FFNs, FFN LayerNorms, residual ordering, remaining attention layers and task head. No 3→1 grouping.

**Default:** final encoder layer 27, 257-token local attention, 180 local cached updates. The selected checkpoint must pass cosine, NMSE and a full-model decision gate; if it fails the native layer is restored. The original 600-step x2 3→1 notebook remains unchanged.

**Important:** This is a research experiment. Quality and GPU speedup are not claimed without running Colab. A faithful 512-token attention sanity check is performed before any training. Holdout is evaluated only once after all selection decisions.

[Open source repository](https://github.com/vtavakkoli/TinyCeNN-LM)


In [ ]:
#@title 2. Setup — ONE attention block, all FFNs frozen (fast)
%pip -q install "transformers==4.57.6" "datasets>=2.20,<5" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"

from pathlib import Path
import copy, gc, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items, normalize_question
)
from tinycenn_lm.decision_training import require_finite
from tinycenn_lm.jev_cenn_full_local_repeat import FaithfulAttentionStage, cpu_state_snapshot
from tinycenn_lm.jev_cenn_pairwindow_superlayer import CellularMultiscale
from tinycenn_lm.laya_lab.core import _valid_tokens

SEED=2028
SOURCE_MODEL="convaiinnovations/laya-typed-decisions" #@param {type:"string"}
TOP_LAYER=27 #@param {type:"integer"}
MAX_LEN=512
LOCAL_WINDOW=257 #@param {type:"integer"}
TRAIN_STEPS=180 #@param {type:"integer"}
CACHE_BATCHES=48 #@param {type:"integer"}
PROBE_BATCHES=12
BATCH=2
PROBE_EVERY=30
LR_ATTENTION=2e-5 #@param {type:"number"}
LR_CENN=3e-4 #@param {type:"number"}
TRAIN_CASES=700
DEV_CASES=220
HOLDOUT_CASES=200
PROBE_CASES=80
DEV_GATE_DECISIONS=240
HOLDOUT_DECISIONS=500
COSINE_WEIGHT=3.0
TARGET_COSINE=0.99
TARGET_NMSE=0.025
TARGET_AGREEMENT=0.99
MAX_GOLD_DROP=0.015
MAX_INCREMENTAL_AGREEMENT_DROP=0.01
USE_FP16_AMP=False
BENCH_REPEATS=8
OUTPUT_DIR=Path("/content/Jev_CeNN_TopAttention_FrozenFFN_Fast")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
assert 0 <= TOP_LAYER < 28 and LOCAL_WINDOW % 2 == 1
assert 0 < LOCAL_WINDOW <= MAX_LEN and TRAIN_STEPS > 0
assert CACHE_BATCHES>=PROBE_BATCHES
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab.")
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda")
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
source_dir=Path(snapshot_download(
    SOURCE_MODEL,allow_patterns=[
      "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"]))
teacher,source_cfg=build_checkpoint_model(source_dir)
student_eval,_=build_checkpoint_model(source_dir)
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
student_eval.encoder.config.reference_compile=False
assert len(teacher.encoder.layers)==28
# The default last layer 27 is a global attention layer.
print("GPU:",torch.cuda.get_device_name(0))
print("Target layer:",TOP_LAYER,"| window:",LOCAL_WINDOW)
print("Full teacher and all 28 encoder FFNs remain unchanged.")


In [ ]:
#@title 3. Partition dataset by CASE: TRAIN, PROBE, DEV-GATE and sealed HOLDOUT
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

from tinycenn_lm.jev_cenn_selective import (
    verify_case_partition, judge_replacement, plan_active_layers,
)
rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="train"))
rng=random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES+DEV_CASES+HOLDOUT_CASES<=len(rows), (
    f"Required TRAIN+DEV+HOLDOUT cases exceeds {len(rows)} dataset TRAIN cases")
train_rows=rows[:TRAIN_CASES]
dev_rows=rows[TRAIN_CASES:TRAIN_CASES+DEV_CASES]
holdout_rows=rows[TRAIN_CASES+DEV_CASES:TRAIN_CASES+DEV_CASES+HOLDOUT_CASES]
split_info=verify_case_partition(train_rows,dev_rows,holdout_rows)
probe_rows=dev_rows[:PROBE_CASES]
gate_rows=dev_rows[PROBE_CASES:]
verify_case_partition(train_rows,probe_rows,gate_rows)
assert not {str(x["id"]) for x in holdout_rows}&{
    str(x["id"]) for x in dev_rows}
train_items=make_items_from_rows(train_rows,SEED,True)
probe_items=make_items_from_rows(probe_rows,SEED+1,True)
gate_items=make_items_from_rows(gate_rows,SEED+2,True)
# IMPORTANT: holdout_items are NOT constructed yet. The final evaluation
# cell materializes these previously sealed rows after all decisions are made.
assert len(gate_items)>=DEV_GATE_DECISIONS
assert len(probe_items)>=PROBE_BATCHES*BATCH
print("Independent partitions:",split_info,
      "probe cases",len(probe_rows),"gate cases",len(gate_rows))
print("Optimizer/probe/gate decisions:",len(train_items),
      len(probe_items),len(gate_items))

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

# Only case-isolated TRAIN / PROBE / DEV-GATE inputs are available here.
# The independent HOLDOUT remains sealed until the final evaluation cell.
print("TRAIN / PROBE / DEV-GATE decisions:",
      len(train_items), len(probe_items), len(gate_items))
print("Official TEST untouched; independent HOLDOUT not materialized.")


In [ ]:
#@title 5. Cache ONLY the input/output of the selected pretrained layer
def check_finite(x,name):
    if not torch.isfinite(x.float()).all(): raise FloatingPointError(name)

def cache_top_layer(items,n_batches):
    chunks=[]
    selected=random.Random(SEED+TOP_LAYER).sample(
        items,min(len(items),n_batches*BATCH))
    for offset in range(0,len(selected)-BATCH+1,BATCH):
        picked=selected[offset:offset+BATCH]
        inp=direct_batch(picked)
        got={}
        def before(mod,args,kwargs):
            got["x"]=args[0].detach()
            got["position_ids"]=kwargs.get("position_ids")
        def after(mod,args,kwargs,res):
            got["end_target"]=res[0].detach()
        h1=teacher.encoder.layers[TOP_LAYER].register_forward_pre_hook(
            before,with_kwargs=True)
        h2=teacher.encoder.layers[TOP_LAYER].register_forward_hook(
            after,with_kwargs=True)
        try:
            with torch.inference_mode():
                teacher(**inp)
        finally:
            h1.remove();h2.remove()
        if not all(k in got for k in ("x","end_target","position_ids")):
            raise RuntimeError("Layer hook missing required tensors")
        data={
            **got,"rollout":got["x"],
            "valid":inp["attention_mask"].bool(),
        }
        chunks.append({
          k:(v.detach().float().cpu().contiguous() if v.is_floating_point()
             else v.detach().cpu().contiguous())
          for k,v in data.items()})
        if len(chunks)%16==0: print(" cached:",len(chunks),"/",n_batches)
    return chunks

def to_cuda(row):
    return {k:v.to(device,non_blocking=True) for k,v in row.items()}

train_cache=cache_top_layer(train_items,CACHE_BATCHES)
probe_cache=cache_top_layer(probe_items,PROBE_BATCHES)
print("Teacher top-layer activations cached:",len(train_cache),"TRAIN,",
      len(probe_cache),"PROBE. No full-model pass during local optimization.")


In [ ]:
#@title 6. Cosine/NMSE and cached independent DEV-GATE teacher predictions
def feature_loss(output,reference,valid):
    pred=output.float()[valid]
    gold=reference.float()[valid]
    # Source activations are validated during cache construction; the
    # differentiable scalar loss is checked once per update. Avoid CUDA
    # sync in these two per-forward tensor checks.
    nmse=(pred-gold).square().sum()/gold.square().sum().clamp_min(1e-8)
    cosine=F.cosine_similarity(pred,gold,dim=-1).mean()
    # Do not call Python float() inside the 520-step CUDA training loop.
    # Converting GPU tensors to floats forces a device synchronization.
    return nmse+COSINE_WEIGHT*(1-cosine),nmse.detach(),cosine.detach()
@torch.inference_mode()
def cached_local_quality(candidate,cache,key="rollout"):
    candidate.eval()
    cs=err=energy=tokens=0.0
    for row in cache:
        e=to_cuda(row)
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            pred=candidate(e[key],attention_mask=e["valid"],
                           position_ids=e["position_ids"])[0]
        a=pred.float()[e["valid"]]
        b=e["end_target"].float()[e["valid"]]
        check_finite(a,"local prediction")
        cs+=float(F.cosine_similarity(a,b,dim=-1).sum())
        err+=float((a-b).square().sum())
        energy+=float(b.square().sum())
        tokens+=int(e["valid"].sum())
    return {"cosine":cs/max(tokens,1),"nmse":err/max(energy,1e-8),
            "tokens":int(tokens)}
def cached_rank(teacher_view,prefix_view):
    return (0.20*(teacher_view["nmse"]+COSINE_WEIGHT*(1-teacher_view["cosine"]))
           +0.80*(prefix_view["nmse"]+COSINE_WEIGHT*(1-prefix_view["cosine"])))
def cache_teacher_predictions(items):
    examples=[]
    for i in range(0,len(items),BATCH):
        inp=direct_batch(items[i:i+BATCH])
        with torch.inference_mode(),torch.autocast(
                "cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            logits,_=teacher(**inp)
        examples.append({
            "batch":inp,
            "teacher":logits.float().detach(),
            "gold":torch.tensor(
                [r["gold_index"] for r in items[i:i+BATCH]],
                device=device,dtype=torch.long)})
    return examples
GATE_ITEMS=gate_items[:DEV_GATE_DECISIONS]
assert len(GATE_ITEMS)==DEV_GATE_DECISIONS
GATE_CACHE=cache_teacher_predictions(GATE_ITEMS)
@torch.inference_mode()
def decision_quality(model,examples):
    model.eval()
    correct=student_gold=teacher_gold=n=0
    js_sum=0.
    for e in examples:
        with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
            logits,_=model(**e["batch"])
        check_finite(logits,"full-model decision logits")
        mask=e["batch"]["marker_mask"].bool()
        a=torch.softmax(e["teacher"].masked_fill(~mask,-1e4),-1)
        b=torch.softmax(logits.float().masked_fill(~mask,-1e4),-1)
        av,bv=a.argmax(-1),b.argmax(-1)
        center=(a+b)/2
        js_sum+=float((0.5*(a*(a.clamp_min(1e-9).log()-
                                  center.clamp_min(1e-9).log())).sum(-1)
                      +0.5*(b*(b.clamp_min(1e-9).log()-
                                  center.clamp_min(1e-9).log())).sum(-1)).sum())
        correct+=int((av==bv).sum())
        student_gold+=int((bv==e["gold"]).sum())
        teacher_gold+=int((av==e["gold"]).sum())
        n+=len(av)
    return {"decisions":n,"agreement":correct/max(n,1),
            "student_gold_accuracy":student_gold/max(n,1),
            "teacher_gold_accuracy":teacher_gold/max(n,1),
            "mean_js":js_sum/max(n,1)}
ORIGINAL_GATE=decision_quality(student_eval,GATE_CACHE)
assert ORIGINAL_GATE["agreement"]==1.
print("Original (no compression) DEV-GATE:",ORIGINAL_GATE)


In [ ]:
#@title 7. Faithful top-layer replacement — QKV, RoPE and FFN preserved
class FrozenFFNAttentionOnlyLayer(nn.Module):
    """
    Replace ONE ModernBERT encoder layer without changing its FFN.
    Identical full-attention baseline when window >= T and CeNN gain=0.
    Final student uses only one attention pass, not four.
    """
    def __init__(self,source_layer,window=257):
        super().__init__()
        self.window=int(window)
        self.attention=FaithfulAttentionStage(source_layer,window=self.window)
        self.attention.norm.requires_grad_(False)  # preserve pretrained attention LayerNorm
        self.ffn_norm=copy.deepcopy(source_layer.mlp_norm)
        self.ffn=copy.deepcopy(source_layer.mlp)
        self.ffn_norm.requires_grad_(False)
        self.ffn.requires_grad_(False)
        hidden=int(source_layer.attn.num_heads*source_layer.attn.head_dim)
        self.cenn=CellularMultiscale(hidden)
        self.cenn_gain=nn.Parameter(torch.tensor(0.0))
        self.max_len=MAX_LEN
        self.architecture="attention_only_original_ffn_frozen_v1"

    def forward(self,hidden_states,attention_mask=None,
                sliding_window_mask=None,position_ids=None,
                cu_seqlens=None,max_seqlen=None,
                output_attentions=False,**kwargs):
        b,t,_=hidden_states.shape
        if t>self.max_len: raise ValueError("sequence length exceeds max")
        valid=_valid_tokens(attention_mask,hidden_states)
        if position_ids is None:
            position_ids=torch.arange(t,device=hidden_states.device)[None,:]
        x=hidden_states
        attn=self.attention(x,valid,position_ids)
        # CeNN correction is zero at initialization and bounded to 10%.
        h=x+attn
        h=h+0.10*torch.tanh(self.cenn_gain)*self.cenn(h,valid)
        # EXACT pretrained, frozen FFN + frozen pretrained normalization.
        h=h+self.ffn(self.ffn_norm(h))
        return (h,None) if output_attentions else (h,)

def ffn_fingerprint(layer):
    return {k:v.detach().cpu().clone() for k,v in layer.ffn.state_dict().items()}

def assert_ffn_untouched(layer,reference):
    actual=layer.ffn.state_dict()
    assert actual.keys()==reference.keys()
    for key in reference:
        assert torch.equal(actual[key].cpu(),reference[key]),key
    assert not any(p.requires_grad for p in layer.ffn.parameters())
    assert not any(p.requires_grad for p in layer.ffn_norm.parameters())

original_layer=student_eval.encoder.layers[TOP_LAYER]
candidate=FrozenFFNAttentionOnlyLayer(
    teacher.encoder.layers[TOP_LAYER],window=LOCAL_WINDOW).to(device)
frozen_ffn=ffn_fingerprint(candidate)
assert_ffn_untouched(candidate,frozen_ffn)
candidate.eval()
# A structural sanity test: full attention + zero CeNN must match native.
sample=to_cuda(probe_cache[0])
with torch.inference_mode():
    # Reference captured inside the original teacher model's actual forward.
    # Calling the native ModernBERT layer directly with a 2D mask would
    # bypass its encoder-prepared attention mask and invalidate this test.
    native_out=sample["end_target"]
    cand_window=candidate.attention.window
    candidate.attention.window=None
    identical=candidate(
        sample["x"],attention_mask=sample["valid"],
        position_ids=sample["position_ids"])[0]
    candidate.attention.window=cand_window
    cosine=float(F.cosine_similarity(
        native_out.float()[sample["valid"]],
        identical.float()[sample["valid"]],dim=-1).mean())
    nmse=float((native_out.float()-identical.float()).square().sum()/
               native_out.float().square().sum().clamp_min(1e-8))
print("UNMODIFIED full-attention reference: cosine",cosine,"NMSE",nmse)
# Stop rather than train a wrong architecture.
assert cosine > 0.999, (
    "New attention implementation is not faithful to teacher; "
    "check native ModernBERT mask/position-id compatibility.")
assert nmse < 0.001, "Full-attention baseline differs from native model."
print("PASS: original residual and frozen FFN topology reproduced.")


In [ ]:
#@title 8. FAST cached training: only TOP_LAYER attention and CeNN
candidate.train()
params_attention=list(candidate.attention.parameters())
params_cenn=list(candidate.cenn.parameters())+[candidate.cenn_gain]
optimizer=torch.optim.AdamW([
 {"params":params_attention,"lr":LR_ATTENTION},
 {"params":params_cenn,"lr":LR_CENN},
],weight_decay=1e-4)
assert_ffn_untouched(candidate,frozen_ffn)
history=[]
best=None
best_rank=float("inf")
best_step=0
rng=random.Random(SEED+TOP_LAYER*77)
order=list(range(len(train_cache)))
idx=0
for step in range(1,TRAIN_STEPS+1):
    if idx==0:rng.shuffle(order)
    row=to_cuda(train_cache[order[idx]])
    idx=(idx+1)%len(order)
    candidate.train()
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda",dtype=amp_dtype,enabled=USE_FP16_AMP):
        out=candidate(row["x"],attention_mask=row["valid"],
                      position_ids=row["position_ids"])[0]
        loss,nmse,cos=feature_loss(out,row["end_target"],row["valid"])
    require_finite(loss,"attention-only distillation")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        params_attention+params_cenn,max_norm=0.5)
    optimizer.step()
    if step==1 or step%PROBE_EVERY==0 or step==TRAIN_STEPS:
        q=cached_local_quality(candidate,probe_cache,"rollout")
        rank=q["nmse"]+COSINE_WEIGHT*(1-q["cosine"])
        history.append({"step":step,"probe":q,"loss":float(loss.detach())})
        if rank<best_rank:
            best_rank=rank;best_step=step
            best=cpu_state_snapshot(candidate)
        print(f"step {step}/{TRAIN_STEPS}: "
              f"probe_cos={q['cosine']:.6f} NMSE={q['nmse']:.6f} "
              f"best_step={best_step}")
assert best is not None
candidate.load_state_dict(best,strict=True)
candidate.eval().requires_grad_(False)
assert_ffn_untouched(candidate,frozen_ffn)
print("Selected cached PROBE checkpoint:",best_step)
del optimizer,best
gc.collect();torch.cuda.empty_cache()


In [ ]:
#@title 9. Strict DEV-GATE: accept only if decisions survive (with rollback)
from tinycenn_lm.jev_cenn_selective import judge_replacement
before=decision_quality(student_eval,GATE_CACHE)
# No other encoder layer, norms, FFN, or head is touched.
student_eval.encoder.layers[TOP_LAYER]=candidate
with torch.inference_mode():
    after=decision_quality(student_eval,GATE_CACHE)
    local=cached_local_quality(candidate,probe_cache,"rollout")
verdict=judge_replacement(
    before,after,
    local_cosine=local["cosine"],local_nmse=local["nmse"],
    min_agreement=TARGET_AGREEMENT,
    max_incremental_agreement_drop=MAX_INCREMENTAL_AGREEMENT_DROP,
    max_gold_drop=MAX_GOLD_DROP,
    min_local_cosine=TARGET_COSINE,
    max_local_nmse=TARGET_NMSE,
    require_speedup=False)
accepted=bool(verdict["accepted"])
if not accepted:
    student_eval.encoder.layers[TOP_LAYER]=original_layer
assert_ffn_untouched(candidate,frozen_ffn)
assert all(not p.requires_grad for p in student_eval.parameters())
print("Native full-model:",before)
print("Candidate full-model:",after)
print("Gate:",verdict)
print("DECISION:", "ACCEPTED" if accepted else "REJECTED, native layer restored")
torch.save({
 "top_layer":TOP_LAYER,"window":LOCAL_WINDOW,"accepted":accepted,
 "architecture":candidate.architecture,
 "state_dict":cpu_state_snapshot(candidate),
 "before":before,"after":after,"gate":verdict,
 "ffn_preserved":True,"training_updates":TRAIN_STEPS
},OUTPUT_DIR/"top_attention_candidate.pt")


In [ ]:
#@title 10. Independent HOLDOUT, GPU timings and reproducible report
@torch.inference_mode()
def gpu_latency(model,inputs,repeats=BENCH_REPEATS):
    model.eval()
    for _ in range(3):model(**inputs)
    torch.cuda.synchronize()
    ms=[]
    for _ in range(repeats):
        start=torch.cuda.Event(enable_timing=True)
        end=torch.cuda.Event(enable_timing=True)
        start.record()
        model(**inputs)
        end.record();end.synchronize()
        ms.append(start.elapsed_time(end))
    return {"median_ms":float(np.median(ms)),
            "p95_ms":float(np.percentile(ms,95)),
            "repeats":repeats}

# Explicit candidate comparison is kept separate from the accepted model.
sample=GATE_CACHE[0]["batch"]
student_eval.encoder.layers[TOP_LAYER]=original_layer
native_latency=gpu_latency(student_eval,sample)
student_eval.encoder.layers[TOP_LAYER]=candidate
candidate_latency=gpu_latency(student_eval,sample)
if not accepted: student_eval.encoder.layers[TOP_LAYER]=original_layer
holdout_items=make_items_from_rows(holdout_rows,SEED+999,False)[:HOLDOUT_DECISIONS]
assert holdout_items,"HOLDOUT has no decisions"
HOLDOUT_CACHE=cache_teacher_predictions(holdout_items)
final_holdout=decision_quality(student_eval,HOLDOUT_CACHE)
report={
 "architecture":candidate.architecture,
 "source_model":SOURCE_MODEL,"top_layer":TOP_LAYER,
 "original_layer_count":len(teacher.encoder.layers),
 "replacement_layer_count":1 if accepted else 0,
 "original_ffn_frozen":True,"original_ffn_preserved":True,
 "attention_window":LOCAL_WINDOW,
 "local_steps":TRAIN_STEPS,"best_step":best_step,
 "training_history":history,
 "local_probe":local,"dev_before":before,"dev_candidate":after,
 "gate":verdict,"accepted":accepted,
 "holdout_untouched_until_final":True,
 "final_accepted_model_holdout":final_holdout,
 "latency_native":native_latency,"latency_candidate":candidate_latency,
 "measured_candidate_speed_ratio":
     native_latency["median_ms"]/max(candidate_latency["median_ms"],1e-9),
 "dataset_case_partition":split_info,
 "limitations":[
   "No independent official TEST used.",
   "The HOLDOUT is disjoint from experiment tuning cases but may overlap with teacher pretraining.",
   "A windowed attention mask can be slower than optimized global SDPA on T4.",
   "Accuracy and latency improvement must be measured, not assumed."
 ]
}
(OUTPUT_DIR/"top_attention_report.json").write_text(
    json.dumps(report,indent=2,allow_nan=False))
print(json.dumps({
  "accepted":accepted,"local_probe":local,
  "holdout":final_holdout,
  "latency_native_ms":native_latency["median_ms"],
  "latency_candidate_ms":candidate_latency["median_ms"],
  "report":str(OUTPUT_DIR/"top_attention_report.json")
},indent=2))
